# Ridge Regression (L2 Regularization) — Notebook README

This notebook demonstrates **Ridge Regression** (linear regression with **$\ell_2$ regularization**) using:

1. `SGDRegressor(penalty='l2')` (gradient-descent-based optimizer in scikit-learn)
2. `Ridge(...)` with an **iterative solver** (`sparse_cg`)
3. A small **from-scratch** implementation that updates parameters using gradient descent

---

## 1) Why Ridge Regression? (Intuition)

Plain Linear Regression can overfit when:
- features are correlated (multicollinearity),
- there are many features relative to samples,
- coefficients become large to fit noise.

**Ridge regression** reduces overfitting by **shrinking** coefficients toward zero (but typically not exactly zero), improving generalization.

---

## 2) Model + Notation

We model:

$$\hat{y} = Xw + b$$

- $X \in \mathbb{R}^{n \times d}$: features
- $w \in \mathbb{R}^{d}$: coefficients
- $b \in \mathbb{R}$: intercept
- $y \in \mathbb{R}^{n}$: target

---

## 3) Loss Functions (Key Formulas)

### Ordinary Least Squares (OLS) / Linear Regression

A common objective (sum of squared errors):

$$J_{\text{OLS}}(w,b) = \sum_{i=1}^{n} (y_i - (x_i^\top w + b))^2$$

(Sometimes you will see the mean squared error with $\frac{1}{n}$ or $\frac{1}{2n}$ scaling; that scaling does **not** change the minimizer, only the gradient magnitude.)

### Ridge Regression (L2 Regularization)

Ridge adds a penalty on the coefficient vector $w$:

$$J_{\text{Ridge}}(w,b) = \sum_{i=1}^{n} (y_i - (x_i^\top w + b))^2 + \alpha \|w\|_2^2$$

Where:
- $\|w\|_2^2 = \sum_{j=1}^{d} w_j^2$
- $\alpha \ge 0$ controls regularization strength

**Important note:** In standard Ridge, the **intercept $b$ is not regularized** (penalty applies to $w$ only).

---

## 4) Closed-Form Solution (When using exact solvers)

If data is centered (or intercept handled separately), the Ridge solution can be written as:

$$w^* = (X^\top X + \alpha I)^{-1}X^\top y$$

This is one reason Ridge is stable when $X^\top X$ is ill-conditioned: the $+\alpha I$ term improves conditioning.

---

## 5) Practical Tips

- **Feature scaling matters** for gradient descent methods. If features are on very different scales, GD/SGD can converge slowly or behave unstably.
- Increasing $\alpha$ typically:
  - decreases variance (less overfitting),
  - increases bias (more underfitting),
  - shrinks coefficients toward 0.
- Evaluate using a metric like **$R^2$** on a held-out test set.

In [1]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
import numpy as np

## Imports

We’ll use:
- `load_diabetes` dataset (regression)
- `train_test_split` to create train/test splits
- `r2_score` as an evaluation metric
- `numpy` for vectorized math (used in the from-scratch GD section)

In [2]:
X,y = load_diabetes(return_X_y=True)

## Load dataset (`load_diabetes`)

`load_diabetes(return_X_y=True)` returns:
- `X`: feature matrix with shape $(n, d)$
- `y`: target vector with shape $(n,)$

The diabetes dataset is already numeric and commonly used for regression demos.

In [3]:
from sklearn.model_selection import train_test_split

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=4)

## Train/test split

We split the dataset into train and test sets:
- Train set: used to fit parameters ($w,b$)
- Test set: used only for evaluation (generalization)

`random_state=4` makes the split reproducible. `test_size=0.2` keeps 20% for testing.

In [5]:
from sklearn.linear_model import SGDRegressor

## Ridge via `SGDRegressor` (Gradient Descent / SGD)

`SGDRegressor` fits linear models using **stochastic gradient descent** (or mini-batch SGD internally).

When you set:
- `penalty='l2'` → adds an $\ell_2$ penalty (Ridge-style regularization)
- `alpha` → controls regularization strength in the SGD objective
- `learning_rate='constant'` and `eta0` → controls step size

### Objective optimized (conceptually)

A typical form is:

$$J(w,b) = \frac{1}{n}\sum_{i=1}^{n} (y_i - (x_i^\top w + b))^2 + \alpha \|w\|_2^2$$

Exact scaling constants can differ across implementations, but the idea is always: **fit error + coefficient shrinkage**.

In [6]:
reg = SGDRegressor(penalty='l2',max_iter=500,eta0=0.1,learning_rate='constant',alpha=0.001)

### Hyperparameters used here (quick guide)

- `max_iter=500`: number of passes over the data
- `eta0=0.1`: initial learning rate (step size)
- `learning_rate='constant'`: keep learning rate fixed at `eta0`
- `alpha=0.001`: L2 regularization strength
- `penalty='l2'`: use Ridge-style penalty

If training is unstable (poor $R^2$), typical fixes are: scaling features, lowering `eta0`, increasing `max_iter`, or tuning `alpha`.

In [7]:
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.38527713095041227
[  50.56853122 -154.70968702  369.84790361  272.29479276   -7.4793536
  -60.93856466 -166.87630858  133.81494771  329.4721628    96.79010416]
[176.71051588]


## Evaluate with $R^2$ score

We evaluate predictions on the test set using the **coefficient of determination** $R^2$:

$$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

- $R^2 = 1$ is perfect prediction
- $R^2 = 0$ is as good as predicting the mean
- $R^2 < 0$ means worse than predicting the mean

Printing `coef_` and `intercept_` shows how Ridge shrinks weights compared to unregularized solutions.

In [8]:
from sklearn.linear_model import Ridge
# In this solver used the Gradient Descent method to find the optimal coefficient s.
reg = Ridge(alpha=0.001, max_iter=500,solver='sparse_cg')

## Ridge via `Ridge` estimator (Iterative solver: `sparse_cg`)

`Ridge` is the dedicated Ridge Regression estimator in scikit-learn.

You chose:
- `solver='sparse_cg'`: **conjugate gradient** method (an iterative optimizer)
- `max_iter=500`: iteration cap for the solver
- `alpha=0.001`: regularization strength

### What does the solver do?

It solves the Ridge optimization problem without explicitly computing a matrix inverse. This can be more efficient for large problems and is numerically stable.

In [9]:
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.4625010162124824
[  34.52193426 -290.84083304  482.4018259   368.06787791 -852.44871823
  501.59161685  180.11114606  270.76335727  759.73535991   37.49136726]
151.10198520510647


## From-scratch Ridge Regression with Gradient Descent (Batch GD)

Below is a minimal **from-scratch** class that tries to fit Ridge Regression using gradient-based updates.

### Standard Ridge objective (recommended)

A common form (with mean scaling) is:

$$J(w,b) = \frac{1}{n}\|y - (Xw + b)\|_2^2 + \alpha\|w\|_2^2$$

Again: **do not regularize $b$** in standard Ridge.

---

### Matrix representation

Let:
- $X \in \mathbb{R}^{n\times d}$, $w \in \mathbb{R}^{d}$, $y \in \mathbb{R}^{n}$
- $\mathbf{1} \in \mathbb{R}^{n}$ is the all-ones vector
- $\hat{y} = Xw + b\mathbf{1}$

Then the objective can be written as:

$$J(w,b)=\frac{1}{n}(y - Xw - b\mathbf{1})^\top (y - Xw - b\mathbf{1}) + \alpha\, w^\top w$$

If you prefer to pack intercept into a single parameter vector, define:

$$\tilde{X} = [\mathbf{1}\ \ X] \in \mathbb{R}^{n\times (d+1)}, \quad \theta = \begin{bmatrix}b \\ w\end{bmatrix}$$

To *exclude the intercept from regularization* in that formulation, use a selector matrix:

$$R = \begin{bmatrix}0 & 0 \\ 0 & I_d\end{bmatrix} \in \mathbb{R}^{(d+1)\times(d+1)}$$

and write:

$$J(\theta)=\frac{1}{n}\|y-\tilde{X}\theta\|_2^2 + \alpha\,\theta^\top R\theta$$

---

### Differentiation of $J$ with respect to the weights $w$

Start from:

$$J(w,b)=\frac{1}{n}(y - Xw - b\mathbf{1})^\top (y - Xw - b\mathbf{1}) + \alpha\, w^\top w$$

Let the residual be:

$$e = y - Xw - b\mathbf{1}$$

So:

$$J(w,b)=\frac{1}{n}e^\top e + \alpha\,w^\top w$$

Differentiate term-by-term:

1) For the data-fit term, using $\nabla_w (e^\top e)=2(\nabla_w e)^\top e$ and $\nabla_w e = -X$:

$$\nabla_w\left(\frac{1}{n}e^\top e\right)=\frac{1}{n}\cdot 2(-X)^\top e = -\frac{2}{n}X^\top e$$

2) For the penalty term:

$$\nabla_w(\alpha\,w^\top w)=2\alpha w$$

Combine:

$$\nabla_w J = -\frac{2}{n}X^\top (y - Xw - b\mathbf{1}) + 2\alpha w$$

Equivalent “prediction minus target” form:

$$\nabla_w J = \frac{2}{n}X^\top (Xw + b\mathbf{1} - y) + 2\alpha w$$

---

### Gradients (compact form)

Let $\hat{y} = Xw + b\mathbf{1}$ and $r = \hat{y} - y$. Then:

$$\nabla_w J = \frac{2}{n}X^\top r + 2\alpha w$$

$$\frac{\partial J}{\partial b} = \frac{2}{n}\sum_{i=1}^{n} r_i$$

### Gradient Descent update

$$w \leftarrow w - \eta\,\nabla_w J$$
$$b \leftarrow b - \eta\,\frac{\partial J}{\partial b}$$

### Note about the implementation below

In the code, parameters are packed into a single vector `thetha` that includes the intercept at index 0. That means the $\ell_2$ term can also affect the intercept in that formulation. Conceptually, pure Ridge excludes the intercept from the penalty; if you want, we can adjust the class later to match the standard form exactly.

In [10]:
class MeraRidgeGD:
    
    def __init__(self,epochs,learning_rate,alpha):
        
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self,X_train,y_train):
        
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        thetha = np.insert(self.coef_,0,self.intercept_)
        
        X_train = np.insert(X_train,0,1,axis=1)
        
        for i in range(self.epochs):
            thetha_der = np.dot(X_train.T,X_train).dot(thetha) - np.dot(X_train.T,y_train) + self.alpha*thetha
            thetha = thetha - self.learning_rate*thetha_der
        
        self.coef_ = thetha[1:]
        self.intercept_ = thetha[0]
    
    def predict(self,X_test):
        
        return np.dot(X_test,self.coef_) + self.intercept_

In [11]:
reg = MeraRidgeGD(epochs=500,alpha=0.001,learning_rate=0.005)

In [12]:
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.4738018280260913
[  46.65050914 -221.3750037   452.12080647  325.54248128  -29.09464178
  -96.47517735 -190.90017011  146.32900372  400.80267299   95.09048094]
150.8697531671347


## Compare results and interpret coefficients

At this point you have three approaches:
- **SGD-based Ridge** (`SGDRegressor` with `penalty='l2'`)
- **Direct Ridge estimator** (`Ridge` with iterative solver)
- **From-scratch GD** (`MeraRidgeGD`)

Things to compare:
- Test **$R^2$** for each method
- Coefficient magnitudes (`coef_`) as you change $\alpha$
- Stability vs speed (GD methods can be sensitive to learning rate and feature scaling)

### Quick experiments
- Increase $\alpha$ (e.g., `0.1`, `1.0`, `10.0`) and observe coefficient shrinkage and $R^2$.
- Try smaller `learning_rate` in the from-scratch implementation if it diverges.

## Summary (Key Takeaways)

- Ridge Regression minimizes **fit error + $\ell_2$ penalty**: $\|y - (Xw+b)\|^2 + \alpha\|w\|^2$.
- Increasing $\alpha$ shrinks coefficients and can reduce overfitting.
- Gradient-based training is sensitive to **feature scaling** and **learning rate**.
- `SGDRegressor(penalty='l2')` and `Ridge(...)` can both produce Ridge-like behavior, but they use different optimization strategies.

If you want, I can also add a small Markdown table that summarizes the hyperparameters used in each approach (SGDRegressor vs Ridge vs from-scratch GD).